# ERA V5 · Session 14 — Dense → MoE upcycling (Colab L4)
Runtime → Change runtime type → **L4 GPU**. Run cells top to bottom.

Everything is written to Google Drive and every (seed, arm) is saved on completion, so after a disconnect just re-run from the top — finished work is skipped.

Plan: 20M-param dense model, 2000 steps → 6 phase-2 arms × 2000 steps (dense_continue, moe_drop, clone_hard, clone_bias, clone_noise, clone_both) × 3 seeds. To shorten, edit `seeds` in `configs/colab20m.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO = '/content/drive/MyDrive/era-v5-session14-moe-upcycling'
os.makedirs(REPO + '/configs', exist_ok=True)
%cd $REPO

In [ ]:
%%writefile model.py
"""Tiny decoder LM: RMSNorm, RoPE, GQA attention, SwiGLU FFN (dense) or MoE FFN.
MoE = 1 always-on shared expert + E routed experts, sigmoid router (fp32), top-k,
aux-loss-free bias balancing (DeepSeek-V3 style), optional annealed Gumbel
("probabilistic") selection for the first steps after upcycling."""
import math, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass, field

@dataclass
class Cfg:
    vocab: int = 2048
    d: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_kv: int = 2          # GQA: 2 KV heads shared by 4 query heads
    seq: int = 128
    ffn: int = 512         # dense SwiGLU hidden
    # MoE (used only when moe=True)
    moe: bool = False
    shared: int = 256      # shared-expert width
    n_exp: int = 8
    exp_w: int = 128
    top_k: int = 2
    route_scale: float = 1.0

class RMSNorm(nn.Module):
    def __init__(s, d): super().__init__(); s.w = nn.Parameter(torch.ones(d))
    def forward(s, x): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * s.w

def rope(T, hd):
    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2).float() / hd))
    f = torch.outer(torch.arange(T).float(), inv)
    return f.cos(), f.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)

class Attn(nn.Module):
    def __init__(s, c):
        super().__init__(); s.c = c; s.hd = c.d // c.n_head
        s.q = nn.Linear(c.d, c.d, bias=False)
        s.kv = nn.Linear(c.d, 2 * c.n_kv * s.hd, bias=False)
        s.o = nn.Linear(c.d, c.d, bias=False)
    def forward(s, x, cos, sin):
        B, T, _ = x.shape; c = s.c
        q = s.q(x).view(B, T, c.n_head, s.hd).transpose(1, 2)
        k, v = s.kv(x).view(B, T, 2, c.n_kv, s.hd).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        rep = c.n_head // c.n_kv
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return s.o(y.transpose(1, 2).reshape(B, T, c.d))

class DenseFFN(nn.Module):
    """SwiGLU: w2( silu(w1 x) * w3 x ). Weight rows of w1/w3 and cols of w2 = 'neurons'."""
    def __init__(s, d, h):
        super().__init__()
        s.w1 = nn.Parameter(torch.randn(h, d) * 0.02)
        s.w3 = nn.Parameter(torch.randn(h, d) * 0.02)
        s.w2 = nn.Parameter(torch.randn(d, h) * 0.02)
    def forward(s, x): return (F.silu(x @ s.w1.t()) * (x @ s.w3.t())) @ s.w2.t()

class MoEFFN(nn.Module):
    def __init__(s, c):
        super().__init__(); s.c = c
        s.shared = DenseFFN(c.d, c.shared) if c.shared > 0 else None
        E, W, d = c.n_exp, c.exp_w, c.d
        s.w1 = nn.Parameter(torch.randn(E, W, d) * 0.02)
        s.w3 = nn.Parameter(torch.randn(E, W, d) * 0.02)
        s.w2 = nn.Parameter(torch.randn(E, d, W) * 0.02)
        s.router = nn.Linear(d, E, bias=False)          # kept fp32 always
        nn.init.normal_(s.router.weight, std=0.006)
        s.register_buffer("bias", torch.zeros(E))         # aux-loss-free balancing bias (selection only)
        s.register_buffer("load", torch.zeros(E))         # token counts since last bias update
        s.register_buffer("win", torch.zeros(E))          # token counts since last stats read (logging)
        s.noise_T = 0.0                                   # >0 => probabilistic (Gumbel) top-k
        s.last_counts = None
    def forward(s, x):
        B, T, d = x.shape; c = s.c
        xf = x.reshape(-1, d)
        with torch.autocast(device_type=x.device.type, enabled=False):   # router always fp32
            logits = s.router(xf.float())
        score = torch.sigmoid(logits)                              # affinity
        sel = score.detach() + s.bias                              # bias only affects WHO is chosen
        if s.training and s.noise_T > 0:
            g = -torch.log(-torch.log(torch.rand_like(sel).clamp_min(1e-9)))
            sel = sel + s.noise_T * g
        idx = sel.topk(c.top_k, -1).indices                        # [N,k]
        w = score.gather(-1, idx)
        w = w / w.sum(-1, keepdim=True) * c.route_scale            # renormalise over chosen (not over bias)
        out = torch.zeros_like(xf)
        counts = torch.bincount(idx.flatten(), minlength=c.n_exp)
        if s.training: s.load += counts.float(); s.win += counts.float()
        s.last_counts = counts
        for e in range(c.n_exp):
            tok, slot = (idx == e).nonzero(as_tuple=True)
            if tok.numel() == 0: continue
            h = xf[tok]
            y = (F.silu(h @ s.w1[e].t()) * (h @ s.w3[e].t())) @ s.w2[e].t()
            out.index_add_(0, tok, (y * w[tok, slot].unsqueeze(-1)).to(out.dtype))
        if s.shared is not None: out = out + s.shared(xf).to(out.dtype)
        return out.view(B, T, d).to(x.dtype)
    @torch.no_grad()
    def update_bias(s, gamma):
        """DeepSeek-V3 aux-loss-free rule, applied over the accumulated window: overloaded -> bias down."""
        if s.load.sum() == 0: return
        f = s.load / s.load.sum()
        s.bias += gamma * torch.sign(f.mean() - f)
        s.load.zero_()

class Block(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.n1, s.n2 = RMSNorm(c.d), RMSNorm(c.d)
        s.attn = Attn(c)
        s.ffn = MoEFFN(c) if c.moe else DenseFFN(c.d, c.ffn)
    def forward(s, x, cos, sin):
        x = x + s.attn(s.n1(x), cos, sin)
        return x + s.ffn(s.n2(x))

class LM(nn.Module):
    def __init__(s, c):
        super().__init__(); s.c = c
        s.emb = nn.Embedding(c.vocab, c.d); nn.init.normal_(s.emb.weight, std=0.02)
        s.blocks = nn.ModuleList(Block(c) for _ in range(c.n_layer))
        s.nf = RMSNorm(c.d)
        cos, sin = rope(c.seq, c.d // c.n_head)
        s.register_buffer("cos", cos, persistent=False); s.register_buffer("sin", sin, persistent=False)
    def forward(s, idx, tgt=None):
        x = s.emb(idx); T = idx.shape[1]
        for b in s.blocks: x = b(x, s.cos[:T], s.sin[:T])
        logits = s.nf(x) @ s.emb.weight.t()                 # tied embeddings
        if tgt is None: return logits
        return F.cross_entropy(logits.view(-1, logits.size(-1)), tgt.reshape(-1))
    def moe_layers(s): return [b.ffn for b in s.blocks if isinstance(b.ffn, MoEFFN)]

def param_counts(m):
    """total vs active (per-token) parameters."""
    total = sum(p.numel() for p in m.parameters())
    active = total
    for f in m.moe_layers():
        per_exp = f.w1[0].numel() + f.w2[0].numel() + f.w3[0].numel()
        active -= (f.c.n_exp - f.c.top_k) * per_exp
    return total, active


In [ ]:
%%writefile upcycle.py
"""Dense -> MoE conversion (the 'growing' step from Session 14).
shared expert : crop the FIRST `shared` neurons of the dense FFN (Rohan's method 1).
routed experts: drop-upcycling (Nakamura et al. 2025, r=0.5): each expert copies a random
                subset of the REMAINING dense neurons and redraws the rest.
Scaling: rest-of-FFN output = sum over R remaining neurons. An expert holding m random
copied neurons, with their w2 scaled by R/m, is an unbiased estimate of that sum; gates are
renormalised to sum to 1 over top-k, so the MoE output is an unbiased estimate of the dense
output at step 0. Redrawn neurons: w1/w3 ~ N(0, std of dense w1/w3); w2 = 0 (zero-init, so
they add no noise at conversion but receive gradient immediately).
mode='clone' : ablation from the lecture's failure case - every expert is an IDENTICAL copy
                (same neuron subset, nothing redrawn)."""
import copy, torch
from model import LM, Cfg, MoEFFN, DenseFFN

@torch.no_grad()
def upcycle(dense: LM, moe_cfg: Cfg, mode="drop", r=0.5, seed=0, redraw_w2="zero"):
    g = torch.Generator().manual_seed(seed)
    moe = LM(moe_cfg)
    sd = {k: v for k, v in dense.state_dict().items() if ".ffn." not in k}
    missing, unexpected = moe.load_state_dict(sd, strict=False)
    assert not unexpected and all(".ffn." in k for k in missing), (missing, unexpected)
    c = moe_cfg
    for bd, bm in zip(dense.blocks, moe.blocks):
        f: DenseFFN = bd.ffn; m: MoEFFN = bm.ffn
        H = f.w1.shape[0]; S = c.shared
        if S:
            m.shared.w1.copy_(f.w1[:S]); m.shared.w3.copy_(f.w3[:S]); m.shared.w2.copy_(f.w2[:, :S])
        rest = torch.arange(S, H); R = len(rest)
        s1, s3, s2 = f.w1[S:].std(), f.w3[S:].std(), f.w2[:, S:].std()
        n_copy = int(round(c.exp_w * (1 - r))) if mode == "drop" else c.exp_w
        n_copy = min(n_copy, R)
        clone_pick = rest[torch.randperm(R, generator=g)[:n_copy]]
        for e in range(c.n_exp):
            pick = clone_pick if mode == "clone" else rest[torch.randperm(R, generator=g)[:n_copy]]
            scale = R / n_copy
            m.w1[e, :n_copy] = f.w1[pick]; m.w3[e, :n_copy] = f.w3[pick]
            m.w2[e, :, :n_copy] = f.w2[:, pick] * scale
            nf = c.exp_w - n_copy
            if nf:
                m.w1[e, n_copy:] = torch.randn(nf, c.d, generator=g) * s1
                m.w3[e, n_copy:] = torch.randn(nf, c.d, generator=g) * s3
                m.w2[e, :, n_copy:] = 0.0 if redraw_w2 == "zero" else torch.randn(c.d, nf, generator=g) * s2
    return moe

@torch.no_grad()
def partition_identity(dense: LM, base: Cfg):
    """Exactness check: split the dense FFN into disjoint experts, turn ALL on with gate 1.
    Must reproduce the dense model bit-for-bit (up to float reassociation)."""
    H = base.ffn; E = 4; assert H % E == 0
    c = Cfg(**{**base.__dict__, "moe": True, "shared": 0, "n_exp": E, "exp_w": H // E, "top_k": E})
    moe = LM(c)
    moe.load_state_dict({k: v for k, v in dense.state_dict().items() if ".ffn." not in k}, strict=False)
    for bd, bm in zip(dense.blocks, moe.blocks):
        f, m = bd.ffn, bm.ffn
        for e in range(E):
            sl = slice(e * H // E, (e + 1) * H // E)
            m.w1[e] = f.w1[sl]; m.w3[e] = f.w3[sl]; m.w2[e] = f.w2[:, sl]
        m.router.weight.zero_()   # equal scores -> gates 1/E each
        m.c.route_scale = E       # -> scale back to 1
    return moe


In [ ]:
%%writefile data.py
"""TinyStories -> own byte-level BPE -> uint16 token arrays.
source='hf'   : datasets.load_dataset (Colab).
source='rows' : HF datasets-server REST API (sandbox, where huggingface.co is blocked)."""
import json, os, time, urllib.request, concurrent.futures as cf, numpy as np

def _rows(split, n):
    url = "https://datasets-server.huggingface.co/rows?dataset=roneneldan/TinyStories&config=default&split={}&offset={}&length=100"
    def get(off):
        for a in range(5):
            try:
                with urllib.request.urlopen(url.format(split, off), timeout=30) as r:
                    return [x["row"]["text"] for x in json.load(r)["rows"]]
            except Exception: time.sleep(2 * (a + 1))
        raise RuntimeError(off)
    with cf.ThreadPoolExecutor(8) as ex: return [t for c in ex.map(get, range(0, n, 100)) for t in c]

def _hf(split, n):
    from datasets import load_dataset
    return load_dataset("roneneldan/TinyStories", split=f"{split}[:{n}]")["text"]

def build(dcfg, vocab, out="data"):
    os.makedirs(out, exist_ok=True)
    tag = f"{dcfg['n_train_stories']}_{vocab}"
    tp, vp, kp = f"{out}/train_{tag}.npy", f"{out}/val_{tag}.npy", f"{out}/tok_{tag}.json"
    if os.path.exists(tp) and os.path.exists(vp): return np.load(tp), np.load(vp)
    from tokenizers import ByteLevelBPETokenizer
    fetch = _hf if dcfg["source"] == "hf" else _rows
    tr = [t.strip() for t in fetch("train", dcfg["n_train_stories"]) if t and t.strip()]
    va = [t.strip() for t in fetch("validation", dcfg["n_val_stories"]) if t and t.strip()]
    tok = ByteLevelBPETokenizer()
    tok.train_from_iterator(tr[:dcfg["tok_train_stories"]], vocab_size=vocab, min_frequency=2, special_tokens=["<|eos|>"])
    tok.save(kp); eos = tok.token_to_id("<|eos|>")
    def enc(texts):
        out_ids = []
        for i in range(0, len(texts), 10000):
            for e in tok.encode_batch(texts[i:i + 10000]): out_ids.extend(e.ids + [eos])
        return np.array(out_ids, dtype=np.uint16)
    a, b = enc(tr), enc(va)
    np.save(tp + ".tmp.npy", a); os.replace(tp + ".tmp.npy", tp)
    np.save(vp + ".tmp.npy", b); os.replace(vp + ".tmp.npy", vp)
    json.dump({"train_tokens": len(a), "val_tokens": len(b), "train_stories": len(tr), "val_stories": len(va),
               "chars_per_token": round(sum(map(len, tr)) / len(a), 3)}, open(f"{out}/stats_{tag}.json", "w"))
    return a, b


In [ ]:
%%writefile train.py
"""Phase 1: dense.  Phase 2 arms from the same checkpoint on identical batches.
python train.py CONFIG SEED ARM   (ARM = phase1 | dense_continue | moe_drop | clone_hard | clone_bias | clone_noise | clone_both)
Outputs go to $OUT (default runs/<config-name>)/seed<k>/."""
import json, os, sys, time, math, torch, numpy as np
from model import LM, Cfg, param_counts
from upcycle import upcycle
import data as D

ARMS = {  # arm -> (conversion mode, bias balancing on, probabilistic selection on)
    "moe_drop": ("drop", True, True), "clone_hard": ("clone", False, False), "clone_bias": ("clone", True, False),
    "clone_noise": ("clone", False, True), "clone_both": ("clone", True, True)}
DEV = "cuda" if torch.cuda.is_available() else "cpu"

class Ctx:
    def __init__(s, cfg_path, seed):
        s.cfg = json.load(open(cfg_path)); s.name = os.path.splitext(os.path.basename(cfg_path))[0]
        s.A = s.cfg["train"]; s.mc = s.cfg["model"]; s.seed = seed
        s.out = os.path.join(os.environ.get("OUT", f"runs/{s.name}"), f"seed{seed}"); os.makedirs(s.out, exist_ok=True)
        tr, va = D.build(s.cfg["data"], s.mc["vocab"], os.environ.get("DATA", "data"))
        s.tr = torch.from_numpy(tr.astype(np.int64)); s.va = torch.from_numpy(va.astype(np.int64))
        s.VAL = [s.batch(0, "val", i) for i in range(s.A["eval_batches"])]
    def model_cfg(s, moe=False): return Cfg(**{**s.mc, "moe": moe})
    def batch(s, step, split="train", i=0):
        d = s.tr if split == "train" else s.va; T = s.mc["seq"]
        g = np.random.default_rng((s.seed, step, i) if split == "train" else (10**9, i))   # val identical for all seeds
        ix = torch.from_numpy(g.integers(0, len(d) - T - 1, s.A["batch"]))
        off = ix[:, None] + torch.arange(T + 1)[None]
        b = d[off]; return b[:, :-1].to(DEV, non_blocking=True), b[:, 1:].to(DEV, non_blocking=True)
    def amp(s): return torch.autocast(DEV, dtype=torch.bfloat16, enabled=bool(s.A.get("bf16")) and DEV == "cuda")
    @torch.no_grad()
    def evaluate(s, m):
        m.eval(); tot = 0.0
        with s.amp():
            for x, y in s.VAL: tot += m(x, y).item()
        m.train(); return tot / len(s.VAL)
    def lr_at(s, g):
        A = s.A; total = A["p1"] + A["p2"]; ds = int(total * (1 - A["decay_frac"]))
        if g < A["warmup"]: return A["lr"] * (g + 1) / A["warmup"]
        return A["lr"] if g < ds else A["lr"] * max(0.0, (total - g) / (total - ds))

def make_opt(m, A):
    dec = [p for p in m.parameters() if p.dim() >= 2]; nod = [p for p in m.parameters() if p.dim() < 2]
    return torch.optim.AdamW([{"params": dec, "weight_decay": A["wd"]}, {"params": nod, "weight_decay": 0.0}],
                             lr=A["lr"], betas=(0.9, 0.95), fused=DEV == "cuda")

def moe_stats(m):
    out = []
    for f in m.moe_layers():
        c = f.win.clone(); f.win.zero_(); fair = c.sum() / len(c)
        out.append(dict(maxvio=round(((c.max() - fair) / fair).item(), 3), dead=int((c < 0.1 * fair).sum()),
                        frac=[round(v, 4) for v in (c / c.sum()).tolist()], bias=[round(v, 3) for v in f.bias.tolist()]))
    return out

def transfer_adam(old_opt, old_m, new_opt, new_m):
    old = dict(old_m.named_parameters()); n = 0
    for k, p in new_m.named_parameters():
        if k in old and old[k].shape == p.shape and old[k] in old_opt.state:
            new_opt.state[p] = {kk: (v.clone() if torch.is_tensor(v) else v) for kk, v in old_opt.state[old[k]].items()}; n += 1
    return n

def run(C, m, opt, start, steps, name, bias_on=False, noise_on=False):
    A = C.A; log = []; moe = bool(m.moe_layers()); t0 = time.time(); ntok = A["batch"] * C.mc["seq"]
    for s in range(start, start + steps):
        for gp in opt.param_groups: gp["lr"] = C.lr_at(s)
        if moe:
            T = A["noise_T0"] * max(0.0, 1 - (s - start) / A["noise_steps"]) if noise_on else 0.0
            for f in m.moe_layers(): f.noise_T = T
        x, y = C.batch(s)
        with C.amp(): loss = m(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if moe and bias_on and (s + 1) % A["bias_every"] == 0:
            for f in m.moe_layers(): f.update_bias(A["gamma"])
        rec = dict(step=s, loss=round(loss.item(), 4))
        if (s + 1) % A["eval_every"] == 0 or s == start:
            rec["val"] = round(C.evaluate(m), 4)
            if moe: rec["moe"] = moe_stats(m)
            el = time.time() - t0; rec["tok_per_s"] = round(ntok * (s - start + 1) / el)
            print(f"[seed{C.seed}] {name} step {s} loss {rec['loss']} val {rec['val']} {rec['tok_per_s']} tok/s", flush=True)
        log.append(rec)
    return log, time.time() - t0

def save_json(obj, path):
    json.dump(obj, open(path + ".tmp", "w")); os.replace(path + ".tmp", path)

def phase1(C):
    torch.manual_seed(C.seed); m = LM(C.model_cfg()).to(DEV); opt = make_opt(m, C.A)
    log, wall = run(C, m, opt, 0, C.A["p1"], "dense_p1")
    torch.save({"model": m.state_dict(), "opt": opt.state_dict()}, f"{C.out}/dense_p1.pt.tmp")
    os.replace(f"{C.out}/dense_p1.pt.tmp", f"{C.out}/dense_p1.pt")
    save_json(dict(params=param_counts(m), wall_s=wall, log=log), f"{C.out}/phase1.json")

def load_dense(C):
    ck = torch.load(f"{C.out}/dense_p1.pt", map_location=DEV)
    d = LM(C.model_cfg()).to(DEV); d.load_state_dict(ck["model"]); o = make_opt(d, C.A); o.load_state_dict(ck["opt"])
    return d, o

def phase2(C, arm):
    torch.manual_seed(C.seed + 1000)
    dense, dopt = load_dense(C); info = {"val_before": C.evaluate(dense)}
    if arm == "dense_continue":
        m, opt, bias_on, noise_on = dense, dopt, False, False
    else:
        mode, bias_on, noise_on = ARMS[arm]
        m = upcycle(dense.cpu(), C.model_cfg(True), mode=mode, r=C.A["r"], seed=C.seed).to(DEV); dense.to(DEV)
        opt = make_opt(m, C.A); info["adam_states_transferred"] = transfer_adam(dopt, dense, opt, m)
        info["val_after_convert"] = C.evaluate(m)
    print(f"[seed{C.seed}] {arm} {info}", flush=True)
    log, wall = run(C, m, opt, C.A["p1"], C.A["p2"], arm, bias_on, noise_on)
    save_json(dict(params=param_counts(m), conversion=info, wall_s=wall, log=log), f"{C.out}/{arm}.json")

if __name__ == "__main__":
    C = Ctx(sys.argv[1], int(sys.argv[2])); arm = sys.argv[3]
    phase1(C) if arm == "phase1" else phase2(C, arm)


In [ ]:
%%writefile conversion_fidelity.py
"""Val loss of the trained dense model vs each conversion variant at step 0 (no training)."""
import json, sys, torch
from train import Ctx, load_dense, DEV
from upcycle import upcycle, partition_identity
def fidelity(C):
    torch.manual_seed(C.seed + 2000); d, _ = load_dense(C); d = d.cpu(); ev = lambda m: round(C.evaluate(m.to(DEV)), 4)
    res = {"partition_all_experts_on": ev(partition_identity(d, C.model_cfg())), "dense": ev(d.cpu())}
    d = d.cpu(); mc = C.model_cfg(True); r = C.A["r"]
    for name, kw in [("drop_r_w2zero", dict(mode="drop", r=r)), ("drop_r_w2normal", dict(mode="drop", r=r, redraw_w2="normal")),
                     ("drop_r0_subset_copy", dict(mode="drop", r=0.0)), ("clone", dict(mode="clone"))]:
        res[name] = ev(upcycle(d, mc, seed=C.seed, **kw))
    json.dump(res, open(f"{C.out}/conversion_fidelity.json", "w"), indent=1); print(f"[seed{C.seed}] fidelity {res}", flush=True)
if __name__ == "__main__": fidelity(Ctx(sys.argv[1], int(sys.argv[2])))


In [ ]:
%%writefile run_all.py
"""Resumable driver: every (seed, arm) whose JSON exists is skipped, so a Colab disconnect costs at most one arm."""
import json, os, sys, time, gc, torch
from train import Ctx, phase1, phase2
from conversion_fidelity import fidelity
cfg = sys.argv[1]; conf = json.load(open(cfg)); t0 = time.time()
for seed in conf["seeds"]:
    C = Ctx(cfg, seed)
    if not os.path.exists(f"{C.out}/phase1.json"): phase1(C)
    if not os.path.exists(f"{C.out}/conversion_fidelity.json"): fidelity(C)
    for arm in conf["arms"]:
        if not os.path.exists(f"{C.out}/{arm}.json"):
            phase2(C, arm); gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        print(f"== done seed{seed} {arm}  elapsed {(time.time()-t0)/60:.1f} min", flush=True)


In [ ]:
%%writefile report.py
"""Aggregate all seeds -> summary.json + plots. README numbers come only from summary.json."""
import json, os, sys, glob, numpy as np, torch, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cfg = sys.argv[1]; conf = json.load(open(cfg)); name = os.path.splitext(os.path.basename(cfg))[0]
root = os.environ.get("OUT", f"runs/{name}"); arms = conf["arms"]; p1 = conf["train"]["p1"]
# only seeds where every arm finished (runs may still be in progress)
seeds = [s for s in conf["seeds"] if all(os.path.exists(f"{root}/seed{s}/{a}.json") for a in ["phase1", "conversion_fidelity"] + arms)]
assert seeds, "no seed has all arms complete"
L = lambda s, f: json.load(open(f"{root}/seed{s}/{f}.json"))
P1 = {s: L(s, "phase1") for s in seeds}; R = {a: {s: L(s, a) for s in seeds} for a in arms}
FID = {s: L(s, "conversion_fidelity") for s in seeds}
vals = lambda log: [(r["step"], r["val"]) for r in log if "val" in r]
r4 = lambda x: round(float(x), 4)
def ms(xs): return {"mean": r4(np.mean(xs)), "std": r4(np.std(xs, ddof=1)) if len(xs) > 1 else 0.0, "per_seed": [r4(x) for x in xs]}
def regain(log, tgt):
    for r in log:
        if "val" in r and r["val"] <= tgt: return r["step"] - p1
    return None
S = {"config": name, "seeds": seeds, "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
     "params_dense": P1[seeds[0]]["params"], "params_moe": R[arms[1]][seeds[0]]["params"] if len(arms) > 1 else None,
     "tokens_per_step": conf["train"]["batch"] * conf["model"]["seq"], "steps": [p1, conf["train"]["p2"]],
     "val_phase1_start": ms([vals(P1[s]["log"])[0][1] for s in seeds]),
     "val_phase1_end": ms([vals(P1[s]["log"])[-1][1] for s in seeds]),
     "fidelity": {k: ms([FID[s][k] for s in seeds]) for k in FID[seeds[0]]},
     "wall_min_total": r4(sum(P1[s]["wall_s"] for s in seeds) / 60 + sum(R[a][s]["wall_s"] for a in arms for s in seeds) / 60),
     "arms": {}}
for a in arms:
    d = {"val_first": ms([vals(R[a][s]["log"])[0][1] for s in seeds]),
         "val_final": ms([vals(R[a][s]["log"])[-1][1] for s in seeds]),
         "steps_to_regain": [regain(R[a][s]["log"], vals(P1[s]["log"])[-1][1]) for s in seeds],
         "tok_per_s": ms([[r for r in R[a][s]["log"] if "tok_per_s" in r][-1]["tok_per_s"] for s in seeds])}
    ev = {s: [r for r in R[a][s]["log"] if "moe" in r] for s in seeds}
    if ev[seeds[0]]:
        d["final_maxvio_mean"] = ms([np.mean([l["maxvio"] for l in ev[s][-1]["moe"]]) for s in seeds])
        d["final_dead_total"] = [sum(l["dead"] for l in ev[s][-1]["moe"]) for s in seeds]
        d["max_dead_any_eval"] = [max(sum(l["dead"] for l in r["moe"]) for r in ev[s][1:]) for s in seeds]
        d["n_experts_total"] = conf["model"]["n_exp"] * conf["model"]["n_layer"]
    S["arms"][a] = d
diff = [vals(R["moe_drop"][s]["log"])[-1][1] - vals(R["dense_continue"][s]["log"])[-1][1] for s in seeds]
S["moe_drop_minus_dense"] = {**ms(diff), "seeds_moe_better": int(sum(x < 0 for x in diff))}
# first eval step after which moe_drop stays below dense (seed-mean)
steps = [st for st, _ in vals(R["dense_continue"][seeds[0]]["log"])]
mm = {a: np.mean([[v for _, v in vals(R[a][s]["log"])] for s in seeds], 0) for a in arms}
below = mm["moe_drop"] < mm["dense_continue"]; cross = None
for i in range(len(steps)):
    if below[i:].all(): cross = steps[i]; break
S["moe_drop_stays_below_dense_from_step"] = cross
json.dump(S, open("summary.json", "w"), indent=1)

col = dict(dense_continue="gray", moe_drop="tab:blue", clone_hard="tab:red", clone_bias="tab:orange", clone_noise="tab:purple", clone_both="tab:green")
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
v1 = np.mean([[v for _, v in vals(P1[s]["log"])] for s in seeds], 0); s1 = [st for st, _ in vals(P1[seeds[0]]["log"])]
ax[0].plot(s1, v1, "k.-", label="dense phase 1")
for a in arms: ax[0].plot(steps, mm[a], ".-", color=col.get(a), label=a)
ax[0].axvline(p1, c="k", ls="--", lw=.8); ax[0].set_title(f"val loss, mean of {len(seeds)} seeds (dashed = conversion)")
ax[0].set_ylim(min(mm[a].min() for a in arms) - .05, v1[len(v1) // 4] + .1); ax[0].legend(); ax[0].set_xlabel("step")
for a in arms:
    lo = np.min([[v for _, v in vals(R[a][s]["log"])] for s in seeds], 0) - mm["dense_continue"]
    hi = np.max([[v for _, v in vals(R[a][s]["log"])] for s in seeds], 0) - mm["dense_continue"]
    ax[1].plot(steps, mm[a] - mm["dense_continue"], ".-", color=col.get(a), label=a)
    if a != "dense_continue": ax[1].fill_between(steps, lo, hi, color=col.get(a), alpha=.12)
ax[1].axhline(0, c="k", lw=.8); ax[1].set_ylim(-0.05, 0.2); ax[1].set_title("val loss minus dense_continue (band = seed min/max)")
ax[1].legend(); ax[1].set_xlabel("step")
for a in arms:
    if a == "dense_continue": continue
    xs = [r["step"] for r in R[a][seeds[0]]["log"] if "moe" in r][1:]
    mv = np.mean([[np.mean([l["maxvio"] for l in r["moe"]]) for r in R[a][s]["log"] if "moe" in r][1:] for s in seeds], 0)
    ax[2].plot(xs, mv, ".-", color=col.get(a), label=a)
ax[2].set_title("expert load imbalance MaxVio (0 = even)"); ax[2].set_xlabel("step"); ax[2].legend()
plt.tight_layout(); plt.savefig("loss_curves.png", dpi=110)
moe_arms = [a for a in arms if a != "dense_continue"]; nl = conf["model"]["n_layer"]
fig, ax = plt.subplots(len(moe_arms), nl, figsize=(2.2 * nl, 1.9 * len(moe_arms)), squeeze=False)
for i, a in enumerate(moe_arms):
    last = [r for r in R[a][seeds[0]]["log"] if "moe" in r][-1]["moe"]
    for l, st in enumerate(last):
        ax[i][l].bar(range(len(st["frac"])), st["frac"], color=col.get(a)); ax[i][l].axhline(1 / len(st["frac"]), c="k", ls="--", lw=.7)
        ax[i][l].set_ylim(0, .6); ax[i][l].set_xticks([]); ax[i][l].set_title(f"{a} L{l} dead={st['dead']}", fontsize=7)
plt.tight_layout(); plt.savefig("expert_load.png", dpi=110)
print(json.dumps(S, indent=1))


In [ ]:
%%writefile make_readme.py
"""README is generated from summary.json so prose numbers cannot drift from artifacts."""
import json
S = json.load(open("summary.json")); A = S["arms"]; n = len(S["seeds"])
f = lambda d: f"{d['mean']:.4f} ± {d['std']:.4f}" if len(d["per_seed"]) > 1 else f"{d['mean']:.4f}"
pd, pm = S["params_dense"], S["params_moe"]
diff = S["moe_drop_minus_dense"]
verdict = ("single seed — direction only, not a claim" if n == 1 else
           "MoE better than dense on every seed" if diff["seeds_moe_better"] == n else
           "MoE worse than dense on every seed" if diff["seeds_moe_better"] == 0 else
           f"mixed: MoE better on {diff['seeds_moe_better']} of {n} seeds — no defensible winner")
rows = "\n".join(f"| {a} | {f(d['val_first'])} | **{f(d['val_final'])}** | {d['steps_to_regain']} | "
                 + (f"{d['final_maxvio_mean']['mean']:.3f} | {d['final_dead_total']} / {d['n_experts_total']} | {d['max_dead_any_eval']}" if "final_maxvio_mean" in d else "– | – | –")
                 + " |" for a, d in A.items())
fid = "\n".join(f"| {k} | {f(v)} |" for k, v in S["fidelity"].items())
open("README.md", "w").write(f"""# ERA V5 · Session 14 — Dense ("linear") model → Mixture-of-Experts

Train a dense model, convert it into an MoE, show it **continues to train** and **loss keeps dropping**.
Everything below is generated from `summary.json` (`make_readme.py`) and asserted by `test_readme.py`.
Config `{S['config']}`, {n} seeds ({S['seeds']}), device `{S['device']}`, total wall time {S['wall_min_total']:.1f} min.

![loss curves](loss_curves.png)

## Setup
| | Dense | MoE after conversion |
|---|---|---|
| Params total / active | {pd[0]:,} / {pd[1]:,} | {pm[0]:,} / {pm[1]:,} |

Model: decoder, RMSNorm, RoPE, GQA, SwiGLU, tied embeddings (`configs/{S['config']}.json`). Data: TinyStories with own
byte-level BPE. {S['tokens_per_step']:,} tokens/step; phase 1 dense {S['steps'][0]} steps, then {S['steps'][1]} steps per arm
from the **same checkpoint on identical batches** with one WSD LR schedule over the whole run. {'bf16 autocast' if S['device'] != 'cpu' else 'fp32 (CPU)'}; router always fp32.

## Conversion (`upcycle.py`)
- **Shared expert** (always on) = crop of the first dense neurons (Rohan's method 1).
- **Routed experts** = drop-upcycling r=0.5: each copies a random half-width subset of the remaining dense neurons,
  redraws the other half (`w1/w3` ~ N(0, dense std), `w2` = 0). Copied `w2` scaled by (remaining/copied) so each expert is
  an unbiased estimate of the remaining dense FFN; gates renormalised to 1 over top-k.
- **Router**: sigmoid, fp32, top-2. **Balancing**: aux-loss-free bias (DeepSeek-V3), selection only, γ=0.01, loads
  accumulated over 8-step windows. **Probabilistic top-k**: annealed Gumbel noise on selection for the first steps.
- AdamW moments carried over for every unchanged tensor (attention, norms, embedding).

## Results (validation loss, {'mean ± std over ' + str(n) + ' seeds' if n > 1 else 'single seed'})
Phase 1: dense val {f(S['val_phase1_start'])} → {f(S['val_phase1_end'])}.

| arm | first eval after conversion | final | steps to regain pre-conversion val (per seed) | final MaxVio | dead experts at end (per seed) | max dead any eval (per seed) |
|---|---|---|---|---|---|---|
{rows}

- **Continues training**: moe_drop goes {f(A['moe_drop']['val_first'])} → {f(A['moe_drop']['val_final'])}.
- **MoE vs dense at equal active params**: final difference moe_drop − dense = {diff['mean']:+.4f}{(' ± ' + format(diff['std'], '.4f')) if n > 1 else ''}
  (per seed {diff['per_seed']}); {verdict}. Seed-mean MoE stays below dense from step {S['moe_drop_stays_below_dense_from_step']}.
- **Ablation — which fix keeps cloned experts alive?** clone_hard (no balancing, no noise) vs clone_bias / clone_noise /
  clone_both isolate the bias balancing and probabilistic selection; moe_drop adds diverse (dropped) initialisation on top.
  Final val per arm: {', '.join(f"{a} {A[a]['val_final']['mean']:.4f}" for a in A)}; dead experts at end:
  {', '.join(f"{a} {A[a]['final_dead_total']}" for a in A if 'final_dead_total' in A[a])}.

- **Key finding**: arms WITH bias balancing ({', '.join(a for a in A if a in ('moe_drop','clone_bias','clone_both'))}) end with
  {[A[a]['final_dead_total'] for a in ('moe_drop','clone_bias','clone_both')]} dead experts and finals
  {[A[a]['val_final']['mean'] for a in ('moe_drop','clone_bias','clone_both')]} vs dense {A['dense_continue']['val_final']['mean']};
  arms WITHOUT it (clone_hard, clone_noise) end with {[A[a]['final_dead_total'] for a in ('clone_hard','clone_noise')]} dead and finals
  {[A[a]['val_final']['mean'] for a in ('clone_hard','clone_noise')]}. Probabilistic selection only delays collapse (0 dead while
  noise is on, then experts die). Drop-upcycling's diverse init did not beat plain clones once balancing was on.

![expert load](expert_load.png)

### Conversion fidelity (val at step 0, no training)
| variant | val |
|---|---|
{fid}

`partition_all_experts_on` equals `dense`: the conversion wiring is exact; every other jump comes from routing and redrawing.

## Limitations
- Small model on TinyStories; {n} seeds only. Differences smaller than the seed std are not claims.
- No MoE-from-scratch baseline at equal compute.
- Run on a 1-core CPU sandbox; `session14_moe_upcycling_colab.ipynb` runs the same pipeline at ~20M params × 3 seeds
  (`configs/colab20m.json`) on a Colab L4 — not yet executed.
- Expert compute is a Python loop over experts (no grouped GEMM), so MoE tok/s understates a real kernel.

## Reproduce
```
pip install tokenizers datasets matplotlib pytest
python -m pytest -q test_invariants.py
python run_all.py configs/{S['config']}.json && python report.py configs/{S['config']}.json && python make_readme.py
python -m pytest -q
```
""")


In [ ]:
%%writefile test_invariants.py
import torch, pytest
from model import LM, Cfg
from upcycle import upcycle, partition_identity

def _dense():
    torch.manual_seed(1); m = LM(Cfg())
    for p in m.parameters(): p.data += 0.02 * torch.randn_like(p)   # non-trivial weights
    return m.eval()

def test_partition_identity_is_exact():
    d = _dense(); m = partition_identity(d, Cfg()).eval()
    x = torch.randint(0, 2048, (2, 64))
    assert torch.allclose(d(x), m(x), atol=1e-5)

def test_non_ffn_weights_copied():
    d = _dense(); m = upcycle(d, Cfg(moe=True))
    for k, v in d.state_dict().items():
        if ".ffn." not in k: assert torch.equal(v, m.state_dict()[k]), k

def test_shared_is_crop_of_first_neurons():
    d = _dense(); m = upcycle(d, Cfg(moe=True)); S = Cfg().shared
    assert torch.equal(m.blocks[0].ffn.shared.w1, d.blocks[0].ffn.w1[:S])

def test_drop_experts_differ_clone_experts_identical():
    d = _dense()
    drop = upcycle(d, Cfg(moe=True), mode="drop").blocks[0].ffn
    clone = upcycle(d, Cfg(moe=True), mode="clone").blocks[0].ffn
    assert not torch.equal(drop.w1[0], drop.w1[1])
    assert all(torch.equal(clone.w1[0], clone.w1[e]) for e in range(8))

def test_redrawn_half_is_half():
    d = _dense(); f = upcycle(d, Cfg(moe=True)).blocks[0].ffn
    assert (f.w2[:, :, 64:] == 0).all() and (f.w2[:, :, :64] != 0).any()

def test_active_params_match_dense():
    from model import param_counts
    dt, _ = param_counts(LM(Cfg())); mt, ma = param_counts(LM(Cfg(moe=True)))
    assert mt > 1.9 * dt and abs(ma - dt) / dt < 0.01   # 2x total, ~same active (router only)

def test_bias_update_direction():
    f = LM(Cfg(moe=True)).blocks[0].ffn
    f.load = torch.tensor([10., 0, 0, 0, 0, 0, 0, 0]); f.update_bias(0.01)
    assert f.bias[0] < 0 and (f.bias[1:] > 0).all()


In [ ]:
%%writefile test_readme.py
"""README must be exactly what make_readme.py produces from summary.json, and summary must match raw runs."""
import json, os, subprocess, sys, shutil
def test_readme_regenerates_identically(tmp_path):
    cur = open("README.md").read(); shutil.copy("README.md", tmp_path / "bak")
    subprocess.run([sys.executable, "make_readme.py"], check=True)
    assert open("README.md").read() == cur
def test_summary_matches_raw_runs():
    S = json.load(open("summary.json")); root = os.environ.get("OUT", f"runs/{S['config']}")
    for s in S["seeds"]:
        for a, d in S["arms"].items():
            log = json.load(open(f"{root}/seed{s}/{a}.json"))["log"]
            assert round([r["val"] for r in log if "val" in r][-1], 4) in d["val_final"]["per_seed"]
def test_conversion_exact():
    S = json.load(open("summary.json")); f = S["fidelity"]
    assert f["partition_all_experts_on"]["per_seed"] == f["dense"]["per_seed"]
def test_moe_keeps_training():
    S = json.load(open("summary.json")); d = S["arms"]["moe_drop"]
    assert all(b < a for a, b in zip(d["val_first"]["per_seed"], d["val_final"]["per_seed"]))
def test_key_finding_prose_holds():
    A = json.load(open("summary.json"))["arms"]
    assert all(sum(A[a]["final_dead_total"]) == 0 for a in ("moe_drop", "clone_bias", "clone_both"))
    assert all(sum(A[a]["final_dead_total"]) > 0 for a in ("clone_hard", "clone_noise"))
    assert all(A[a]["val_final"]["mean"] < A["dense_continue"]["val_final"]["mean"] for a in ("moe_drop", "clone_bias", "clone_both"))
    assert min(A["clone_bias"]["val_final"]["mean"], A["clone_both"]["val_final"]["mean"]) <= A["moe_drop"]["val_final"]["mean"]


In [ ]:
%%writefile .gitignore
data/
runs/*/seed*/dense_p1.pt
__pycache__/
.pytest_cache/


In [ ]:
%%writefile configs/colab20m.json
{"model": {"vocab": 4096, "d": 384, "n_layer": 8, "n_head": 6, "n_kv": 2, "seq": 256, "ffn": 1536,
           "shared": 768, "n_exp": 8, "exp_w": 384, "top_k": 2, "route_scale": 1.0},
 "data":  {"source": "hf", "n_train_stories": 400000, "n_val_stories": 5000, "tok_train_stories": 100000},
 "train": {"batch": 64, "lr": 0.0015, "warmup": 200, "p1": 2000, "p2": 2000, "decay_frac": 0.2, "wd": 0.1,
           "eval_every": 100, "eval_batches": 20, "gamma": 0.01, "bias_every": 8, "noise_T0": 0.05,
           "noise_steps": 400, "r": 0.5, "bf16": true},
 "seeds": [0, 1, 2],
 "arms": ["dense_continue", "moe_drop", "clone_hard", "clone_bias", "clone_noise", "clone_both"]}


In [ ]:
%%writefile configs/cpu_tiny.json
{
 "model": {
  "vocab": 2048,
  "d": 128,
  "n_layer": 4,
  "n_head": 4,
  "n_kv": 2,
  "seq": 128,
  "ffn": 512,
  "shared": 256,
  "n_exp": 8,
  "exp_w": 128,
  "top_k": 2,
  "route_scale": 1.0
 },
 "data": {
  "source": "rows",
  "n_train_stories": 40000,
  "n_val_stories": 2000,
  "tok_train_stories": 20000
 },
 "train": {
  "batch": 32,
  "lr": 0.002,
  "warmup": 100,
  "p1": 1500,
  "p2": 1500,
  "decay_frac": 0.2,
  "wd": 0.1,
  "eval_every": 100,
  "eval_batches": 40,
  "gamma": 0.01,
  "bias_every": 8,
  "noise_T0": 0.05,
  "noise_steps": 300,
  "r": 0.5,
  "bf16": false
 },
 "seeds": [
  0,
  1,
  2
 ],
 "arms": [
  "dense_continue",
  "moe_drop",
  "clone_hard",
  "clone_bias",
  "clone_noise",
  "clone_both"
 ]
}

In [ ]:
%%writefile configs/smoke.json
{
 "model": {
  "vocab": 2048,
  "d": 128,
  "n_layer": 4,
  "n_head": 4,
  "n_kv": 2,
  "seq": 128,
  "ffn": 512,
  "shared": 256,
  "n_exp": 8,
  "exp_w": 128,
  "top_k": 2,
  "route_scale": 1.0
 },
 "data": {
  "source": "rows",
  "n_train_stories": 3000,
  "n_val_stories": 300,
  "tok_train_stories": 3000
 },
 "train": {
  "batch": 32,
  "lr": 0.002,
  "warmup": 10,
  "p1": 40,
  "p2": 40,
  "decay_frac": 0.2,
  "wd": 0.1,
  "eval_every": 20,
  "eval_batches": 4,
  "gamma": 0.01,
  "bias_every": 4,
  "noise_T0": 0.05,
  "noise_steps": 20,
  "r": 0.5,
  "bf16": false
 },
 "seeds": [
  0,
  1
 ],
 "arms": [
  "dense_continue",
  "moe_drop",
  "clone_hard",
  "clone_bias",
  "clone_noise",
  "clone_both"
 ]
}

In [ ]:
!pip -q install tokenizers datasets matplotlib pytest
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# conversion-logic invariants (CPU-cheap, ~5 s)
!python -m pytest -q test_invariants.py

## Main run
First run downloads TinyStories and trains the tokenizer (~several min). Watch `tok/s` in the first lines to estimate total time. If Colab disconnects: re-run all cells; completed arms are skipped.

In [ ]:
!python -u run_all.py configs/colab20m.json 2>&1 | tee -a run_colab20m.log

In [ ]:
!python report.py configs/colab20m.json > /dev/null && python make_readme.py && python -m pytest -q
from IPython.display import Image, Markdown, display
display(Image('loss_curves.png')); display(Image('expert_load.png')); display(Markdown(open('README.md').read()))

## Ship
Download the zip, or push from here (replace the URL; needs a GitHub token).

In [ ]:
!cd .. && zip -qr era-v5-session14-moe-upcycling.zip era-v5-session14-moe-upcycling -x '*/data/*' '*.pt' && ls -la era-v5-session14-moe-upcycling.zip
# !git init && git add -A && git commit -m 'Session 14: dense -> MoE upcycling' && git branch -M main
# !git remote add origin https://<TOKEN>@github.com/rjvim/era-v5-session14-moe-upcycling.git && git push -u origin main